[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-07-testing-pipelines.ipynb#scrollTo=ff000001)

---
# Day 7 · Testing Hamilton Pipelines
**certified-journeys / hamilton-certified** · Day 7 · Testing

> **Goal for today:** Master three levels of Hamilton testing — unit tests (no Driver), config-branch tests (Driver with injected config), and integration tests (full pipeline with `assert_equivalent_dataframes`).

In [ ]:
%pip install -q sf-hamilton pytest

## The Three Levels of Hamilton Testing

| Level | What you test | Needs Driver? | Speed |
|---|---|---|---|
| **Unit** | One function in isolation | No | Fast |
| **Config branch** | Both sides of `@config.when` | Yes | Medium |
| **Integration** | Full pipeline end-to-end | Yes | Slower |

The key advantage of Hamilton: **unit tests require zero infrastructure**. Every function is pure Python — pass a Series in, assert on the Series out. No mocking, no fixtures, no database.

We'll build a small pipeline to test, then exercise all three levels.

In [ ]:
import sys, types
import numpy as np
import pandas as pd
from hamilton import driver
from hamilton.function_modifiers import tag, config, extract_columns, check_output
from hamilton.plugins import h_pandas

# ── Pipeline under test ────────────────────────────────────────────────────────

@extract_columns('age', 'spend', 'tenure_months', 'has_complaint')
def raw_features(raw_data: pd.DataFrame) -> pd.DataFrame:
    return raw_data[['age', 'spend', 'tenure_months', 'has_complaint']]

@check_output(data_type=pd.Series, allow_nans=False)
def spend_cleaned(spend: pd.Series) -> pd.Series:
    """Replace negative spend with 0."""
    return spend.clip(lower=0)

@check_output(data_type=pd.Series, allow_nans=False)
def age_cleaned(age: pd.Series) -> pd.Series:
    """Clamp age to [18, 100], fill NaN with median."""
    return age.clip(18, 100).fillna(age.median())

@tag(feature_type='numerical')
def tenure_years(tenure_months: pd.Series) -> pd.Series:
    return tenure_months / 12.0

@tag(feature_type='numerical')
def spend_log(spend_cleaned: pd.Series) -> pd.Series:
    return np.log1p(spend_cleaned)

@tag(feature_type='boolean')
def is_high_spender(spend_cleaned: pd.Series) -> pd.Series:
    return (spend_cleaned > spend_cleaned.quantile(0.75)).astype(float)

@tag(feature_type='boolean')
def is_at_risk(has_complaint: pd.Series, spend_log: pd.Series) -> pd.Series:
    """At-risk: has complaint AND below-median spend."""
    return ((has_complaint == 1) & (spend_log < spend_log.median())).astype(float)

# Two config branches: 'simple' uses raw age; 'normalized' z-scores it
@config.when(age_mode='simple')
def age_feature__simple(age_cleaned: pd.Series) -> pd.Series:
    """Pass-through: use raw cleaned age."""
    return age_cleaned

@config.when(age_mode='normalized')
def age_feature__normalized(age_cleaned: pd.Series) -> pd.Series:
    """Z-score normalized age."""
    return (age_cleaned - age_cleaned.mean()) / age_cleaned.std()

# Package into a module
pipeline = types.ModuleType('pipeline')
for fn in [raw_features, spend_cleaned, age_cleaned, tenure_years,
           spend_log, is_high_spender, is_at_risk,
           age_feature__simple, age_feature__normalized]:
    setattr(pipeline, fn.__name__, fn)
sys.modules['pipeline'] = pipeline

# Synthetic test data
rng = np.random.default_rng(7)
test_df = pd.DataFrame({
    'age':            rng.integers(18, 75, 100).astype(float),
    'spend':          np.where(rng.random(100) < 0.1, -5.0, rng.exponential(60, 100)),
    'tenure_months':  rng.integers(0, 60, 100).astype(float),
    'has_complaint':  rng.integers(0, 2, 100).astype(float),
})
print('Pipeline and test data ready. Bad spend rows:', (test_df['spend'] < 0).sum())

## Step 1 · Level 1: Unit Tests — No Driver Required

Call each Hamilton function directly with crafted inputs. The goal: **test the exact logic, not the plumbing**.

Good unit tests for Hamilton functions:
- Use the smallest possible inputs that exercise the logic
- Test boundary conditions (zero, null, extreme values)
- Assert on the output's exact values, not just its shape

In [ ]:
# ── Unit tests: spend_cleaned ──────────────────────────────────────────────────
def test_spend_cleaned_removes_negatives():
    result = spend_cleaned(pd.Series([-10.0, 0.0, 50.0, -1.0]))
    assert result.tolist() == [0.0, 0.0, 50.0, 0.0]

def test_spend_cleaned_leaves_positive_unchanged():
    s = pd.Series([1.0, 100.0, 200.0])
    pd.testing.assert_series_equal(spend_cleaned(s), s)

def test_spend_cleaned_handles_zero():
    assert spend_cleaned(pd.Series([0.0])).iloc[0] == 0.0

# ── Unit tests: age_cleaned ────────────────────────────────────────────────────
def test_age_cleaned_clamps_high():
    result = age_cleaned(pd.Series([150.0, 200.0, 25.0]))
    assert result.max() <= 100.0

def test_age_cleaned_clamps_low():
    result = age_cleaned(pd.Series([5.0, 10.0, 30.0]))
    assert result.min() >= 18.0

def test_age_cleaned_fills_nulls():
    result = age_cleaned(pd.Series([25.0, None, 35.0]))
    assert not result.isna().any()

# ── Unit tests: is_high_spender ────────────────────────────────────────────────
def test_is_high_spender_threshold_at_75th_percentile():
    # [10, 20, 30, 40] → 75th pct = 32.5 → only 40 is high
    result = is_high_spender(pd.Series([10.0, 20.0, 30.0, 40.0]))
    assert result.tolist() == [0.0, 0.0, 0.0, 1.0]

# ── Unit tests: tenure_years ───────────────────────────────────────────────────
def test_tenure_years_converts_correctly():
    result = tenure_years(pd.Series([0.0, 6.0, 12.0, 24.0]))
    assert result.tolist() == [0.0, 0.5, 1.0, 2.0]

# ── Unit tests: is_at_risk ─────────────────────────────────────────────────────
def test_is_at_risk_requires_both_conditions():
    # Only row 0 has complaint=1 AND below-median spend_log
    complaints  = pd.Series([1.0, 1.0, 0.0, 0.0])
    spend_log_s = pd.Series([1.0, 5.0, 1.0, 5.0])  # median = 3.0
    result = is_at_risk(complaints, spend_log_s)
    assert result.tolist() == [1.0, 0.0, 0.0, 0.0]

# Run all unit tests
tests = [
    test_spend_cleaned_removes_negatives,
    test_spend_cleaned_leaves_positive_unchanged,
    test_spend_cleaned_handles_zero,
    test_age_cleaned_clamps_high,
    test_age_cleaned_clamps_low,
    test_age_cleaned_fills_nulls,
    test_is_high_spender_threshold_at_75th_percentile,
    test_tenure_years_converts_correctly,
    test_is_at_risk_requires_both_conditions,
]
for t in tests:
    t()
    print(f'  ✓ {t.__name__}')
print(f'\nAll {len(tests)} unit tests passed!')

### What just happened?
- **9 tests, zero Driver, zero fixtures** — every test is a direct function call with a hand-crafted `pd.Series`.
- **Boundary conditions**: zero spend, null age, exact quantile boundary for `is_high_spender`.
- **`is_at_risk` test**: both conditions are independently varied — confirms the AND logic, not just one branch.

## Step 2 · Level 2: Config Branch Tests

For `@config.when` functions, build two Drivers — one per branch — and verify each activates correctly. This catches the common mistake of branches having different semantics than intended.

In [ ]:
OUTPUTS = ['age_feature', 'spend_log', 'is_high_spender', 'tenure_years']
inputs  = {'raw_data': test_df}

# Build one Driver per config branch
dr_simple = (
    driver.Builder()
    .with_modules(pipeline)
    .with_config({'age_mode': 'simple'})
    .build()
)
dr_norm = (
    driver.Builder()
    .with_modules(pipeline)
    .with_config({'age_mode': 'normalized'})
    .build()
)

result_simple = dr_simple.execute(OUTPUTS, inputs=inputs)
result_norm   = dr_norm.execute(OUTPUTS, inputs=inputs)

# Test: simple branch should return cleaned age values (range [18, 100])
assert result_simple['age_feature'].min() >= 18.0
assert result_simple['age_feature'].max() <= 100.0
print('✓ simple branch: age_feature is in [18, 100]')

# Test: normalized branch should have mean ≈ 0 and std ≈ 1
assert abs(result_norm['age_feature'].mean()) < 0.01
assert abs(result_norm['age_feature'].std() - 1.0) < 0.01
print('✓ normalized branch: age_feature has mean≈0, std≈1')

# Test: other outputs should be identical across branches (age_mode doesn't affect them)
pd.testing.assert_series_equal(
    result_simple['spend_log'], result_norm['spend_log'], check_names=False
)
print('✓ spend_log is identical across both branches (not affected by age_mode)')

### What just happened?
- **Two Drivers, one module** — the only difference is `with_config({'age_mode': ...})`.
- **Branch isolation test**: confirmed that `spend_log` is unaffected by `age_mode` — catches accidental coupling.
- **Semantic test on each branch**: not just "it ran" but "it produced the right kind of output".

## Step 3 · Level 3: Integration Tests with `overrides`

Integration tests verify that **multiple nodes work correctly together**. In Hamilton, `overrides` let you inject a known mid-graph value to isolate the downstream subgraph — making integration tests fast and deterministic.

In [ ]:
dr_int = (
    driver.Builder()
    .with_modules(pipeline)
    .with_config({'age_mode': 'simple'})
    .build()
)

# Integration test: given a known spend_cleaned, does is_at_risk behave correctly?
# We override spend_cleaned so we control its exact values
known_spend_cleaned = pd.Series([5.0, 5.0, 200.0, 200.0])  # first two are low
known_complaints    = pd.Series([1.0, 0.0, 1.0, 0.0])

result = dr_int.execute(
    ['is_at_risk', 'spend_log', 'is_high_spender'],
    inputs={'raw_data': pd.DataFrame({
        'age': [30.0]*4, 'spend': [0.0]*4,   # raw spend — will be overridden
        'tenure_months': [12.0]*4, 'has_complaint': known_complaints,
    })},
    overrides={'spend_cleaned': known_spend_cleaned}  # bypass spend_cleaned
)

# spend_log of [5, 5, 200, 200] → log1p gives ≈ [1.79, 1.79, 5.30, 5.30]
# median ≈ 3.55 → rows 0,1 are below-median (low spend)
# is_at_risk: complaint=1 AND below-median spend_log → only row 0
assert result['is_at_risk'].tolist() == [1.0, 0.0, 0.0, 0.0], \
    f"Expected [1,0,0,0], got {result['is_at_risk'].tolist()}"
print('✓ Integration test: is_at_risk correctly identifies only row 0 as at-risk')
print('  spend_log values:', result['spend_log'].round(2).tolist())
print('  is_at_risk:      ', result['is_at_risk'].tolist())

### What just happened?
- **`overrides={'spend_cleaned': ...}`** injected a known value, making the test deterministic regardless of the raw data or `spend_cleaned` logic.
- We tested the `spend_log → is_at_risk` interaction in isolation — without this override, a change to `spend_cleaned` could mask a bug in `is_at_risk`.
- This is the Hamilton equivalent of mocking a dependency — but without any mock framework.

## Step 4 · Graph Validation Tests

Beyond testing output values, you can test **the graph structure itself** — assert that certain nodes exist, have the right types, or carry specific tags. These tests catch regressions when the graph is refactored.

In [ ]:
dr_validate = (
    driver.Builder()
    .with_modules(pipeline)
    .with_config({'age_mode': 'simple'})
    .build()
)
nodes = {v.name: v for v in dr_validate.list_available_variables()}

# Test: required nodes exist
required_nodes = ['spend_cleaned', 'age_cleaned', 'tenure_years',
                  'spend_log', 'is_high_spender', 'is_at_risk', 'age_feature']
for n in required_nodes:
    assert n in nodes, f'Missing required node: {n}'
print(f'✓ All {len(required_nodes)} required nodes present')

# Test: tagged nodes have correct feature_type
assert nodes['spend_log'].tags.get('feature_type') == 'numerical'
assert nodes['is_high_spender'].tags.get('feature_type') == 'boolean'
print('✓ spend_log tagged as numerical')
print('✓ is_high_spender tagged as boolean')

# Test: all nodes have type annotations (a common oversight)
untyped = [n for n, v in nodes.items() if v.type is None]
assert not untyped, f'Nodes missing type annotations: {untyped}'
print('✓ All nodes have type annotations')

# Test: upstream/downstream relationships are correct
upstream_of_is_at_risk = [n.name for n in dr_validate.what_is_upstream_of('is_at_risk')]
assert 'spend_log' in upstream_of_is_at_risk
assert 'has_complaint' in upstream_of_is_at_risk
print('✓ is_at_risk correctly depends on spend_log and has_complaint')

### What just happened?
- **Graph validation tests** don't execute the pipeline — they inspect the DAG structure.
- **Tag assertions** prevent silent tag removal during refactoring — if someone removes `@tag(feature_type='boolean')`, the test catches it.
- **Upstream assertions** confirm the dependency wiring is correct — a refactor that accidentally disconnects `spend_log` from `is_at_risk` would fail here.

## Step 5 · Full Pipeline Integration Test

The final test: run the pipeline end-to-end with known input data and assert on the full output DataFrame — shape, dtypes, value ranges, and no nulls.

In [ ]:
from hamilton.plugins import h_pandas

dr_full = (
    driver.Builder()
    .with_modules(pipeline)
    .with_config({'age_mode': 'simple'})
    .with_adapters(h_pandas.PandasDataFrameResult())
    .build()
)

ALL_FEATURES = ['age_feature', 'spend_log', 'tenure_years', 'is_high_spender', 'is_at_risk']
output = dr_full.execute(ALL_FEATURES, inputs={'raw_data': test_df})

# Shape
assert output.shape == (len(test_df), len(ALL_FEATURES)), \
    f'Expected {(len(test_df), len(ALL_FEATURES))}, got {output.shape}'
print(f'✓ Output shape: {output.shape}')

# No nulls in any feature
null_counts = output.isna().sum()
assert null_counts.sum() == 0, f'Unexpected nulls: {null_counts[null_counts > 0]}'
print('✓ No null values in any feature')

# Boolean features are 0.0 or 1.0 only
for col in ['is_high_spender', 'is_at_risk']:
    assert set(output[col].unique()).issubset({0.0, 1.0}), f'{col} has non-binary values'
print('✓ Boolean features contain only 0.0 and 1.0')

# spend_log is non-negative (log1p of non-negative input)
assert (output['spend_log'] >= 0).all()
print('✓ spend_log is non-negative')

# age_feature is in [18, 100] for simple mode
assert output['age_feature'].between(18, 100).all()
print('✓ age_feature is clamped to [18, 100]')

print(f'\nAll integration tests passed! Output:\n{output.describe().round(3)}')

### What just happened?
- **End-to-end integration test** covers shape, nullability, dtype semantics, and value ranges.
- These assertions would catch: a clean step accidentally introducing NaNs, a boolean feature returning floats outside {0,1}, or the adapter producing the wrong number of columns.
- Together, the three test levels form a **testing pyramid**: many unit tests, a few branch tests, one integration test.

In [ ]:
# Challenge: add a graph validation test that asserts:
# 1. The pipeline has at least 5 nodes tagged feature_type='numerical' or 'boolean'
# 2. 'spend_cleaned' is upstream of 'is_high_spender'
# 3. No node name contains 'TODO' (guard against accidental placeholder names)

# nodes = {v.name: v for v in dr_validate.list_available_variables()}
# ...

print('Implement the three graph validation assertions!')

---
## Day 7 key concepts recap

| Level | Tool | When to use |
|---|---|---|
| Unit | Direct function call | Logic correctness, boundary conditions |
| Branch | Two Drivers with different `with_config()` | Verify `@config.when` semantics per branch |
| Integration | `overrides` + full Driver execute | Multi-node interactions, deterministic pipelines |
| Graph validation | `list_available_variables`, `what_is_upstream_of` | Structural regressions, tag correctness |

> **Tip:** Each Hamilton function is a pure Python function — test it exactly like any other. No mocking, no fixtures needed for unit tests. Just call it with inputs and assert on the output.

---
## What's next
**Day 8** → Advanced decorators: `@pipe` for chained transformations, `@mutate` for post-processing, and `@resolve` for dynamic node inclusion.

Mark Day 7 complete in your [tracker](../index.html).